# Hierarchische Sequenzmodelle (LSTM/GRU)

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** Die Überarbeitung aus #173 im hierarchischen Ansatz prüfen (#194). Bisher kam LSTM → LSTM auf 0,568 (70/30). Im kombinierten Ansatz brachte die Überarbeitung +0,13 (#174).

**Konfigurationen** (Raster aus `35_jd_sequenzmodelle`, nur was dort gewonnen hat):

| Konfiguration | Kultur | Stadium | Eingabe |
| --- | --- | --- | --- |
| SVM → SVM | SVM | SVM | alle Bänder (Referenz aus #163) |
| Netz → Netz, Sentinel-2 | GRU/LSTM | GRU/LSTM | 10 Sensorbänder, AEZ/Monat-Embeddings |
| Netz → Netz, alle Bänder | GRU/LSTM | GRU/LSTM | 8 Bänder je Schritt, Embeddings |
| SVM → GRU, Sentinel-2 | SVM | GRU | Standardkontext (One-Hot), weil beide Ebenen dieselbe Vorverarbeitung bekommen |

Die Kultur-Stufe ist laut #163 der Engpass. Deshalb ist die Mischung mit SVM für die Kultur interessant.

**Vorgehen:** CV auf 80/20 (Experiment `crop-stage`, approach `hierarchical`), danach die beste Netz-Variante **einmal** validiert.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.svm import SVC

from awp2.config import (
    LSTM_EMBEDDINGS_CONTEXT,
    LSTM_STANDARD_CONTEXT,
    MODEL_DOCS_FIGURES_DIR,
    REDUCTION_BEST_PARAMS_COL,
    SENTINEL2_SENSOR,
)
from awp2.evaluation import METRIC_LABELS
from awp2.experiment import RunConfig, TuneConfig, run
from awp2.models import (
    HierarchicalClassifier,
    SpectralLSTMClassifier,
    make_hierarchical_spectral_lstm,
    make_hierarchical_svm,
)
from awp2.plots import save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import CURVE_METRICS, ReductionVariant, reduction_curve

warnings.filterwarnings("ignore", message="Training stopped at max_epochs")

APPROACH = "hierarchical"
spectra = PreprocessingConfig(use_spectral_standard_scale=True)
embedded = spectra.model_copy(update={"use_categorical_context": True})
embedded_s2 = embedded.model_copy(update={"spectral_sensor": SENTINEL2_SENSOR})
spectra_s2 = spectra.model_copy(update={"spectral_sensor": SENTINEL2_SENSOR})


def net_grid(level: str, **choices: list[Any]) -> dict[str, list[Any]]:
    fixed = {
        "hidden_size": [64],
        "num_layers": [1],
        "dropout": [0.2],
        "head_hidden_size": [64],
        "wavelength_channel": [True],
        "max_epochs": [200],
    }
    return {f"{level}__{key}": values for key, values in {**fixed, **choices}.items()}


def both_levels(**choices: list[Any]) -> dict[str, list[Any]]:
    # the crop and stage nets share one setting, keeping the grid small
    return {**net_grid("crop_estimator", **choices), **net_grid("stage_estimator", **choices)}


svm_then_gru = HierarchicalClassifier(
    SVC(class_weight="balanced"),
    SpectralLSTMClassifier(context_mode=LSTM_STANDARD_CONTEXT, cell="gru"),
)
configs: dict[str, tuple[BaseEstimator, PreprocessingConfig, dict[str, list[Any]]]] = {
    "svm_svm": (
        make_hierarchical_svm(),
        spectra,
        {
            "crop_estimator__C": [30.0],
            "crop_estimator__gamma": [0.01],
            "stage_estimator__C": [100.0],
            "stage_estimator__gamma": [0.003],
        },
    ),
    "net_net_sentinel2": (
        make_hierarchical_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
        embedded_s2,
        both_levels(cell=["gru", "lstm"], bands_per_step=[1], learning_rate=[0.003]),
    ),
    "net_net_all_bands": (
        make_hierarchical_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
        embedded,
        both_levels(cell=["lstm", "gru"], bands_per_step=[8], learning_rate=[0.001]),
    ),
    "svm_gru_sentinel2": (
        svm_then_gru,
        spectra_s2,
        {
            "crop_estimator__C": [30.0],
            "crop_estimator__gamma": ["scale"],
            **net_grid("stage_estimator", bands_per_step=[1], learning_rate=[0.003]),
        },
    ),
}

## 1. CV-Vergleich

In [ ]:
def evaluate_config(
    name: str,
    model: BaseEstimator,
    preprocessing: PreprocessingConfig,
    grid: dict[str, list[Any]],
) -> pd.DataFrame:
    return reduction_curve(
        model,
        [ReductionVariant(name=name, family="all_bands", preprocessing=preprocessing)],
        TuneConfig(
            name=f"hier_seq_{name}",
            approach=APPROACH,
            description=f"Hierarchische Sequenzmodelle (#194): {name}",
            param_grid=grid,
            preprocessing=preprocessing,
        ),
    )


results = pd.concat(
    [evaluate_config(name, *config) for name, config in configs.items()], ignore_index=True
)
results[["variant", *CURVE_METRICS, "bacc_combined_std", REDUCTION_BEST_PARAMS_COL]].round(3)

In [ ]:
labels = {
    "svm_svm": "SVM → SVM",
    "net_net_sentinel2": "Netz → Netz, S2",
    "net_net_all_bands": "Netz → Netz, alle Bänder",
    "svm_gru_sentinel2": "SVM → GRU, S2",
}
table = results.set_index("variant").loc[list(labels)]
fig, ax = plt.subplots(figsize=(10, 4.5))
table[list(CURVE_METRICS)].rename(index=labels, columns=METRIC_LABELS).plot.bar(
    ax=ax,
    yerr=table[[f"{m}_std" for m in CURVE_METRICS]].to_numpy().T,
    rot=0,
    capsize=2,
    width=0.8,
)
ax.set_ylabel("BAcc (CV)")
ax.set_title("Hierarchisch: Sequenzmodelle vs. SVM")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize="small", loc="lower left")
save_doc_figure(fig, "hierarchisch_sequenzmodelle_cv", MODEL_DOCS_FIGURES_DIR)

**Befund CV:**

| Konfiguration | Kultur | Stadium | kombiniert |
| --- | --- | --- | --- |
| SVM → SVM | 0,938 | 0,902 | **0,846** ± 0,018 |
| SVM → GRU, Sentinel-2 | 0,930 | 0,851 | 0,805 ± 0,008 |
| GRU → GRU, Sentinel-2 | 0,898 | 0,852 | 0,768 ± 0,039 |
| GRU → LSTM, alle Bänder | 0,874 | 0,820 | 0,733 ± 0,042 |

- **Die Überarbeitung wirkt auch hierarchisch:** Das beste reine Netz-Modell (GRU → GRU auf Sentinel-2) kommt auf 0,768. Bisher erreichte LSTM → LSTM 0,546 CV bzw. 0,568 Validierung (70/30).
- **Sentinel-2 ist wieder besser als alle Bänder** (0,768 gegenüber 0,733). Wie im kombinierten Ansatz lernen die Netze kürzere Sequenzen leichter.
- **GRU schlägt LSTM auf der Kultur-Stufe** (alle Bänder: 0,73 mit GRU-Kultur gegenüber 0,70 mit LSTM-Kultur). Beim Stadium ist der Zelltyp egal.
- **Die Mischung SVM → GRU ist das beste Sequenzmodell hierarchisch** (0,805, kleinste Streuung). Die SVM erkennt die Kultur fast so gut wie in der reinen SVM-Hierarchie (0,930). Der Verlust liegt ganz beim Stadium: Das GRU kommt auf 0,851 statt 0,902.
- **Warum das Stadium leidet:** Jedes Stage-Netz lernt nur aus den Zeilen seiner Kultur, bei Reis sind das 75. Für ein Netz ist das deutlich zu wenig, die SVM kommt damit besser zurecht.

## 2. Einmalige Validierung

Die beste Netz-Variante nach CV. Referenz: hierarchische SVM `hier_reduction_all_bands` (`514156e7`, Val. 0,845).

In [ ]:
network_variants = ["net_net_sentinel2", "net_net_all_bands", "svm_gru_sentinel2"]
candidates = results[results["variant"].isin(network_variants)]
winner = candidates.loc[candidates["bacc_combined"].idxmax()]
model, preprocessing, _grid = configs[winner["variant"]]
result = run(
    model.set_params(**winner[REDUCTION_BEST_PARAMS_COL]),
    RunConfig(
        name=f"hierarchical_{winner['variant']}",
        approach=APPROACH,
        description=f"Bestes hierarchisches Sequenzmodell nach CV (#194): {winner['variant']}",
        tuning_run=winner["tuning_run"],
        preprocessing=preprocessing,
    ),
)
pd.Series(
    {**result.metrics.model_dump(), "run_id": result.run_id[:8], "variant": winner["variant"]}
)

## Fazit

| | CV kombiniert | Val. Kultur | Val. Stadium | Val. kombiniert | Val. Macro-F1 kombiniert |
| --- | --- | --- | --- | --- | --- |
| SVM → SVM (`514156e7`) | 0,846 | 0,918 | 0,905 | **0,845** | 0,827 |
| SVM → GRU, Sentinel-2 (`8be59dc8`) | 0,805 | 0,930 | 0,851 | 0,799 | 0,763 |
| bisher LSTM → LSTM (`hierarchical_lstm_embeddings`, 70/30) | 0,546 | 0,812 | 0,778 | 0,568 | 0,529 |
| *kombiniert: GRU, Sentinel-2 (`b6d2362b`)* | 0,794 | 0,908 | 0,890 | 0,832 | 0,773 |

- **Das hierarchische Sequenzmodell verbessert sich um 0,23** auf der Validierung (0,568 → 0,799). Den größten Anteil hat die SVM auf der Kultur-Stufe.
- **Die SVM bleibt in beiden Stufen vorn.** Ein Netz für das Stadium pro Kultur hat zu wenige Trainingszeilen.
- **Im kombinierten Ansatz funktioniert das Netz besser** (0,832 gegenüber 0,799): Dort lernt ein einziges Netz aus allen rund 4.500 Zeilen.
- **Für M3:** Auch hierarchisch profitieren die Netze von der Datenreduktion. Sentinel-2 ist hier besser als alle Bänder.
- **Endmodell:** Hier ändert sich nichts. Die kombinierte RBF-SVM bleibt vorn.